<h1 style='font-weight:bold; color:#003366; text-align:center;'>Tema 12: Ingeniería de variables (Feature engineering)</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

<h2 style='font-weight:bold; color:#003366;'>1. Transformación de variables para modelado</h2>

**Feature engineering** es construir las columnas con las que se va a modelar: combinar, dividir,
agrupar y marcar, a partir de las columnas que vinieron. Es el trabajo de traducir conocimiento del negocio
a columnas de una tabla que un algoritmo pueda utilizar para buscar patrones.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path


def ruta_datos(nombre_archivo):
    """Busca el archivo en las ubicaciones habituales del curso, incluida Colab."""
    candidatas = [
        Path("../Datos") / nombre_archivo,
        Path("Datos") / nombre_archivo,
        Path("/content/Datos") / nombre_archivo,
        Path(nombre_archivo),
    ]
    for candidata in candidatas:
        if candidata.exists():
            return str(candidata)
    raise FileNotFoundError(f"No se encontró {nombre_archivo}.")


def tasa_por_decil(datos, variable, objetivo="mora_90_dias", cantidad=10):
    """Tasa del evento por decil de la variable, con los casos y el valor mediano de cada decil."""
    validos = datos[[variable, objetivo]].dropna()
    decil = pd.qcut(validos[variable], cantidad, labels=False, duplicates="drop") + 1
    tabla = validos.groupby(decil).agg(
        casos=(objetivo, "size"),
        valor_mediano=(variable, "median"),
        tasa_mora_pct=(objetivo, lambda columna: columna.mean() * 100),
    )
    tabla.index.name = f"decil de {variable}"
    return tabla.round({"valor_mediano": 3, "tasa_mora_pct": 1})


solicitudes = pd.read_csv(ruta_datos("credito_clientes.csv")).drop_duplicates().reset_index(drop=True)

print(f"Solicitudes: {len(solicitudes):,}")
print(f"Tasa de mora a 90 días: {solicitudes['mora_90_dias'].mean():.2%}")
display(tasa_por_decil(solicitudes, "ingreso_mensual"))

tabla_ingreso = tasa_por_decil(solicitudes, "ingreso_mensual")

print(f"\nPoder de separación del ingreso (max sobre min): {tabla_ingreso['tasa_mora_pct'].max() / tabla_ingreso['tasa_mora_pct'].min():.2f}")

<h2 style='font-weight:bold; color:#003366;'>2. Ratios</h2>

Un **ratio** es un cociente entre dos columnas. Es la familia de variables derivadas más productiva en
finanzas y la razón es conceptual: **un ratio es una magnitud relativa, y las decisiones financieras son
relativas.** Una deuda de un millón no dice nada; un millón sobre un patrimonio de diez es una cosa y
sobre un patrimonio de uno es otra.

Los ratios que valen la pena en una cartera de crédito:

- **Cuota / ingreso mensual**: Carga financiera mensual, explica qué proporción del sueldo se va en la cuota.
- **Monto / ingreso anual**: Endeudamiento respecto de la capacidad de pago, expresa cuántos meses de sueldo es el crédito.
- **Cuota / ingreso disponible**: Carga sobre lo que queda después de los gastos fijos (más exigente que el primero).

Algo importante a tener en cuenta: **todo ratio tiene un denominador, y el denominador tiene
ceros y faltantes.** Dividir por cero da infinito, dividir por un faltante da faltante, y dividir por un
número negativo cambia el signo y manda la fila al lugar equivocado de la tabla.

In [ ]:
# La cuota no está en el archivo: se calcula como en el sistema de la empresa
RECARGO_ADMINISTRATIVO = 1.09

solicitudes["cuota_estimada"] = (solicitudes["monto_solicitado"] / solicitudes["plazo_meses"]
                                 * RECARGO_ADMINISTRATIVO)

solicitudes["ratio_cuota_ingreso"] = (solicitudes["cuota_estimada"] / solicitudes["ingreso_mensual"])

tabla_ratio = tasa_por_decil(solicitudes, "ratio_cuota_ingreso")
display(tabla_ratio)
print(f"\nPoder de separación del ratio cuota/ingreso (max sobre min): {tabla_ratio['tasa_mora_pct'].max() / tabla_ratio['tasa_mora_pct'].min():.2f}")
print(f"Ratio mediano de la cartera: {solicitudes['ratio_cuota_ingreso'].median():.1%} del ingreso")

In [ ]:
CANASTA_MINIMA = 20_000  # gasto de subsistencia que el área de riesgo supone para un hogar

solicitudes["ingreso_anual"] = solicitudes["ingreso_mensual"] * 12
solicitudes["ingreso_disponible"] = solicitudes["ingreso_mensual"] - CANASTA_MINIMA

solicitudes["monto_sobre_ingreso_anual"] = (solicitudes["monto_solicitado"]
                                            / solicitudes["ingreso_anual"])
solicitudes["ratio_cuota_disponible"] = (solicitudes["cuota_estimada"]
                                         / solicitudes["ingreso_disponible"])

print("Monto sobre ingreso anual:")
display(tasa_por_decil(solicitudes, "monto_sobre_ingreso_anual"))

print("\nCuota sobre ingreso disponible, tal como sale la división:")
display(tasa_por_decil(solicitudes, "ratio_cuota_disponible"))

sin_disponible = solicitudes["ingreso_disponible"] <= 0
print(f"\nSolicitudes con ingreso disponible menor o igual a cero: {sin_disponible.sum()} "
      f"({sin_disponible.mean():.1%})")
print(f"Tasa de mora de esas solicitudes: "
      f"{solicitudes.loc[sin_disponible, 'mora_90_dias'].mean():.1%}")
print(f"Ratio mínimo de la columna: {solicitudes['ratio_cuota_disponible'].min():.1f}")

**Conclusiones:** 

- `monto_sobre_ingreso_anual` nos permite ver que el crédito equivalente a más de medio año de sueldo se atrasa casi el doble que el equivalente a un mes.

- `ratio_cuota_disponible` tiene un problema en el denominador.
El primer decil tiene una **mediana negativa** (-0,80) y una tasa de mora del 23,3%, más alta que la de los seis deciles
que le siguen. Son las 335 solicitudes cuyo ingreso no llega a la canasta mínima: al restarles la
canasta, el denominador queda negativo, el ratio da negativo, y `qcut` los manda al decil de los "menos
cargados", que es exactamente el lugar opuesto al que les corresponde. 

Las tres formas de tratar un denominador problemático, y hay que elegir a propósito:

1. Denominador cero o negativo: Marcarlo aparte con un indicador y dejar el ratio en `NaN`. 
2. Denominador faltante: El ratio queda `NaN`. Reportar sobre cuántos casos se calculó.
3. Denominador muy chico: Acotar el ratio en un percentil alto, diciendo cuántas filas afectó.



<h2 style='font-weight:bold; color:#003366;'>3. Indicadores de faltante: que falte también es información</h2>

En el tema 9 vimos que el faltante puede tener **patrón**. Si tiene patrón, entonces el hecho de que
falte dice algo sobre el caso, y eso se puede convertir en una columna:

```python
datos["ingreso_no_informado"] = datos["ingreso_mensual"].isna().astype(int)
```

In [ ]:
solicitudes["ingreso_no_informado"] = solicitudes["ingreso_mensual"].isna().astype(int)
solicitudes["nivel_no_informado"] = solicitudes["nivel_educativo"].isna().astype(int)
solicitudes["antiguedad_no_informada"] = solicitudes["antiguedad_laboral"].isna().astype(int)

INDICADORES = ["ingreso_no_informado", "nivel_no_informado", "antiguedad_no_informada"]

for indicador in INDICADORES:
    resumen = solicitudes.groupby(indicador).agg(
        casos=("mora_90_dias", "size"),
        tasa_mora_pct=("mora_90_dias", lambda columna: columna.mean() * 100),
    ).round(1)
    diferencia = (resumen.loc[1, "tasa_mora_pct"] - resumen.loc[0, "tasa_mora_pct"])
    display(resumen)
    print(f"Diferencia en la mora: {diferencia:+.1f} puntos porcentuales sobre "
          f"{resumen.loc[1, 'casos']:.0f} casos\n")

<h2 style='font-weight:bold; color:#003366;'>4. Discretizar: de continuo a tramos</h2>

Discretizar es convertir una variable continua en tramos nos puede ayudar cuando:

- El efecto **no es lineal**. Si la mora salta a partir del 40% de carga financiera y antes es plana, un
  tramo lo captura y una recta no.
- El tramo tiene **significado de negocio**. 
- La variable tiene atípicos que no querés dejar afuera: el último tramo los absorbe.
- El resultado va a una tabla que lee alguien que no es analista.

Las dos herramientas que sirven para discretizar son:

- `pd.cut(columna, bins=[...])`: los cortes los pone **el negocio**. Cada tramo tiene el ancho que
  querés y la cantidad de casos que caiga.
- `pd.qcut(columna, 5)`: los cortes los pone **la distribución**. Cada tramo tiene el 20% de los casos y
  el ancho lo decide el dato.

Para fijar una política se usa `cut`, porque un umbral que se mueve cada vez que llegan datos nuevos no
es una política. Para describir la cartera se usa `qcut`.

In [ ]:
# cut: los cortes salen del manual de crédito
CORTES_CARGA = [0, 0.15, 0.30, 0.45, np.inf]
NOMBRES_CARGA = ["Con Margen", "Normal", "Ajustado", "Crítico"]

solicitudes["tramo_carga"] = pd.cut(solicitudes["ratio_cuota_ingreso"],
                                    bins=CORTES_CARGA, labels=NOMBRES_CARGA)

por_tramo = solicitudes.groupby("tramo_carga", observed=True).agg(
    casos=("mora_90_dias", "size"),
    ratio_mediano=("ratio_cuota_ingreso", "median"),
    tasa_mora_pct=("mora_90_dias", lambda columna: columna.mean() * 100),
)
por_tramo["participacion_pct"] = por_tramo["casos"] / por_tramo["casos"].sum() * 100
print("cut: cortes de negocio")
display(por_tramo.round(2))

# qcut: cada quintil tiene el 20% de los casos
solicitudes["quintil_carga"] = pd.qcut(solicitudes["ratio_cuota_ingreso"], 5,
                                       labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
por_quintil = solicitudes.groupby("quintil_carga", observed=True).agg(
    casos=("mora_90_dias", "size"),
    ratio_maximo=("ratio_cuota_ingreso", "max"),
    tasa_mora_pct=("mora_90_dias", lambda columna: columna.mean() * 100),
)
print("\nqcut: cortes por cantidad de casos")
display(por_quintil.round(2))

<h2 style='font-weight:bold; color:#003366;'>5. Agregaciones por grupo como variable</h2>

Hasta acá cada variable nueva salía de la misma fila. Otra familia entera sale de comparar la fila
**contra su grupo**: cuánto se aparta el ingreso de este cliente de la mediana de su sector de empleo,
cuánto se aparta la compra de esta sucursal del promedio de su región, cuánto se aparta el gasto de este
mes del promedio del cliente.

La herramienta es `groupby().transform()` que calcula un estadístico por grupo y
lo devuelve **para cada fila de la tabla**, en lugar de una fila por grupo.

```python
mediana_del_sector = datos.groupby("sector_empleo")["ingreso_mensual"].transform("median")
datos["ingreso_relativo"] = datos["ingreso_mensual"] / mediana_del_sector
```

Un valor de 1,3 quiere decir "gana 30% más que la mediana de su sector". Es una variable distinta del
ingreso: dos personas con el mismo sueldo pueden estar una arriba y otra abajo de su grupo.

In [ ]:
mediana_del_sector = solicitudes.groupby("sector_empleo")["ingreso_mensual"].transform("median")
solicitudes["ingreso_relativo_sector"] = solicitudes["ingreso_mensual"] / mediana_del_sector

print("Mediana de ingreso por sector de empleo:")
display(solicitudes.groupby("sector_empleo")["ingreso_mensual"].median().round(0))

print("\nTasa de mora por quintil de ingreso relativo al sector:")
display(tasa_por_decil(solicitudes, "ingreso_relativo_sector", cantidad=5))

print("\nY para comparar, la misma tabla con el ingreso sin relativizar:")
display(tasa_por_decil(solicitudes, "ingreso_mensual", cantidad=5))

**Conclusión: esta variable no sirvió** La tasa de mora por quintil de ingreso
relativo al sector se comporta de forma muy parecida que el ingreso general.

Las medianas de ingreso de los cuatro sectores son casi idénticas
(entre $U 35.972 y $U 38.942). **No hay información nueva** por lo que esta variable no aporta información, esto solo es útil cuando los grupos son heterogéneos.

<h2 style='font-weight:bold; color:#003366;'>6. Interacciones: cruzar dos categóricas</h2>

Una **interacción** es el efecto conjunto de dos variables, que no es la suma de sus efectos separados.
En finanzas es frecuente: el alquiler puede ser un problema para un independiente y no para un empleado
público, porque lo que cambia es la estabilidad del ingreso, no el alquiler.

Cruzar dos categóricas se hace con `pd.crosstab`, y hay que mirar **dos tablas siempre**: la de tasas y
la de cantidad de casos. 

In [ ]:
tasas = pd.crosstab(solicitudes["sector_empleo"], solicitudes["situacion_vivienda"],
                    values=solicitudes["mora_90_dias"], aggfunc="mean") * 100
casos = pd.crosstab(solicitudes["sector_empleo"], solicitudes["situacion_vivienda"])

print("Tasa de mora por combinación, en porcentaje:")
display(tasas.round(1))
print("\nCantidad de solicitudes en cada celda:")
display(casos)

print(f"\nCelda con menos casos: {casos.values.min()} solicitudes")

<h2 style='font-weight:bold; color:#003366;'>7. Codificar categóricas y escalar</h2>

Los cálculos y los algoritmos trabajan con números, así que las categorías hay que traducirlas. **Cómo
se traduce depende de si la categoría tiene orden o no**.

**Nominales (sin orden):** departamento, sector de empleo, canal. Se usa `pd.get_dummies`, que crea una
columna de 0 y 1 por categoría. Por qué no se pueden numerar: si a Montevideo le pones 1, a Canelones 2 y
a Salto 3, le estás diciendo al modelo que **Canelones está entre Montevideo y Salto** y que la distancia
entre Montevideo y Salto es el doble que entre Montevideo y Canelones. Eso no significa nada, y el modelo
lo va a tomar en serio.

**Ordinales (con orden):** nivel educativo, tramo de carga financiera, calificación de riesgo. Acá el
orden **es información real** y numerarlas es lo correcto: Secundaria 1, Terciaria incompleta 2,
Universitaria 3, Posgrado 4. Se hace con un diccionario explícito y `map`, nunca dejando que pandas
ordene alfabéticamente, porque alfabéticamente Posgrado va antes que Secundaria.

In [ ]:
# Nominales: una columna de 0 y 1 por categoría
NOMINALES = ["sector_empleo", "situacion_vivienda"]
indicadoras = pd.get_dummies(solicitudes[NOMINALES], dtype=int)
print(f"{len(NOMINALES)} columnas nominales se transforman en {indicadoras.shape[1]} columnas de 0 y 1")
display(indicadoras.head(3))

# Ordinales: el orden es información, así que se declara a mano
ORDEN_EDUCATIVO = {
    "Secundaria": 1,
    "Terciaria incompleta": 2,
    "Universitaria": 3,
    "Posgrado": 4,
}
solicitudes["nivel_educativo_orden"] = solicitudes["nivel_educativo"].map(ORDEN_EDUCATIVO)

print("\nTasa de mora según el nivel educativo, en el orden correcto:")
display(solicitudes.groupby("nivel_educativo_orden").agg(
    casos=("mora_90_dias", "size"),
    tasa_mora_pct=("mora_90_dias", lambda columna: columna.mean() * 100)).round(1))

print("\nLo que pasaría si dejáramos que ordene el alfabeto:")
print(sorted(ORDEN_EDUCATIVO))

**Conclusiones:** Dos columnas nominales se convierten en ocho columnas de 0 y 1. Es el precio de no
inventar un orden: la tabla se ensancha. Con una variable de 50 categorías (por ejemplo, comercio
adherido) `get_dummies` genera 50 columnas y ahí ya conviene agrupar las categorías chicas en "Otros"
antes de codificar.

En el nivel educativo el orden importa y se ve: la mora baja de 22,6% en Secundaria a 17,9% en Posgrado,
pero **no de forma monótona** (Terciaria incompleta, con 23,5%, está peor que Secundaria). Ese quiebre es
interpretable: "terciaria incompleta" suele significar que la carrera se abandonó, muchas veces por
problemas económicos.


<h2 style='font-weight:bold; color:#003366;'>8. Modelado (ejemplo, no entra en el curso)</h2>

1. **Separar** los datos en entrenamiento y prueba, antes de calcular cualquier cosa (sección 8).
2. **Ajustar** un modelo con los datos de entrenamiento. Vamos a usar una **regresión logística**, que es
   una regresión que en lugar de predecir un número predice la probabilidad de que pase algo.
3. **Evaluar** sobre los datos de prueba, que el modelo no vio.

Vamos a ajustar la misma regresión dos veces: una con las columnas que vinieron en el archivo y otra
agregando las que construimos hoy.

> ### Antes de modelar
>
> En esta cartera, el 21,5% de los créditos entra en mora y el 78,5% no.
>
> El modelo simple que dice **"nadie se va a atrasar"**, siempre, sin mirar
> ninguna columna. **¿Qué porcentaje de los casos va a acertar ese modelo?**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split

VARIABLES_DEL_ARCHIVO = ["edad", "ingreso_mensual", "antiguedad_laboral", "monto_solicitado",
                         "plazo_meses", "tarjetas_activas", "consultas_ultimos_6m"]
VARIABLES_CONSTRUIDAS = VARIABLES_DEL_ARCHIVO + [
    "ratio_cuota_ingreso", "monto_sobre_ingreso_anual", "ingreso_no_informado",
    "nivel_educativo_orden", "ingreso_relativo_sector",
]

objetivo = solicitudes["mora_90_dias"]
entrenamiento, prueba, objetivo_entrenamiento, objetivo_prueba = train_test_split(
    solicitudes, objetivo, test_size=0.25, random_state=2026, stratify=objetivo)

print(f"Entrenamiento: {len(entrenamiento):,} solicitudes, "
      f"mora {objetivo_entrenamiento.mean():.1%}")
print(f"Prueba: {len(prueba):,} solicitudes, mora {objetivo_prueba.mean():.1%}")
print("\nstratify mantiene la proporción de morosos igual en las dos partes: "
      "con clases desbalanceadas eso no es opcional.")

In [ ]:
def ajustar_y_evaluar(columnas):
    medianas = entrenamiento[columnas].median()
    promedios = entrenamiento[columnas].fillna(medianas).mean()
    desvios = entrenamiento[columnas].fillna(medianas).std()

    x_entrenamiento = (entrenamiento[columnas].fillna(medianas) - promedios) / desvios
    x_prueba = (prueba[columnas].fillna(medianas) - promedios) / desvios

    # class_weight="balanced" pesa igual las dos clases; sin eso el modelo aprende
    # a decir "nadie se atrasa", que es lo que acabamos de discutir.
    modelo = LogisticRegression(max_iter=1000, class_weight="balanced")
    modelo.fit(x_entrenamiento, objetivo_entrenamiento)

    probabilidades = modelo.predict_proba(x_prueba)[:, 1]
    predicciones = (probabilidades >= 0.5).astype(int)
    return {
        "variables": len(columnas),
        "exactitud": accuracy_score(objetivo_prueba, predicciones),
        "precision": precision_score(objetivo_prueba, predicciones, zero_division=0),
        "sensibilidad": recall_score(objetivo_prueba, predicciones),
        "auc": roc_auc_score(objetivo_prueba, probabilidades),
    }, predicciones


metricas_archivo, predicciones_archivo = ajustar_y_evaluar(VARIABLES_DEL_ARCHIVO)
metricas_construidas, predicciones_construidas = ajustar_y_evaluar(VARIABLES_CONSTRUIDAS)

resultados = pd.DataFrame([
    {"modelo": "Nadie se atrasa", "variables": 0,
     "exactitud": 1 - objetivo_prueba.mean(), "precision": 0.0,
     "sensibilidad": 0.0, "auc": 0.5},
    {"modelo": "Variables del archivo", **metricas_archivo},
    {"modelo": "Con variables construidas", **metricas_construidas},
])
print(resultados.round(3).to_string(index=False))

**Interpretación, en el orden en que hay que leerla.**

**Primero, la predicción.** El modelo que dice "nadie se atrasa" tiene una **exactitud del 78,5%** y no
sirve absolutamente para nada: no identifica ni un moroso. Su sensibilidad es cero. Ese es el punto de
partida obligatorio de cualquier evaluación con clases desbalanceadas: **la exactitud de un modelo
inútil ya es alta**, así que la exactitud sola no dice si el modelo sirve.

**Segundo, la comparación que es el punto de la clase.** Misma regresión logística, mismos datos, misma
separación entre entrenamiento y prueba. Lo único que cambia son las columnas

In [ ]:
matriz = confusion_matrix(objetivo_prueba, predicciones_construidas)
etiquetas = pd.DataFrame(
    matriz,
    index=["Real: no se atrasó", "Real: se atrasó"],
    columns=["Predicho: no se atrasa", "Predicho: se atrasa"],
)
print(etiquetas.to_string())

verdaderos_negativos, falsos_positivos, falsos_negativos, verdaderos_positivos = matriz.ravel()
print(f"\nMorosos detectados: {verdaderos_positivos} de "
      f"{verdaderos_positivos + falsos_negativos} "
      f"({verdaderos_positivos / (verdaderos_positivos + falsos_negativos):.1%})")
print(f"Morosos que se escaparon: {falsos_negativos}")
print(f"Buenos clientes rechazados por error: {falsos_positivos} de "
      f"{verdaderos_negativos + falsos_positivos} "
      f"({falsos_positivos / (verdaderos_negativos + falsos_positivos):.1%})")
print(f"\nExactitud del modelo: {accuracy_score(objetivo_prueba, predicciones_construidas):.1%}")
print(f"Exactitud de 'nadie se atrasa': {1 - objetivo_prueba.mean():.1%}")

Las cuatro métricas:

| Métrica | Qué contesta | Cuándo importa |
|---|---|---|
| **Exactitud** | Qué proporción de los casos acertó | Casi nunca sola. Engaña con clases desbalanceadas |
| **Precisión** | De los que marqué como morosos, cuántos lo eran | Cuando marcar de más cuesta plata o clientes |
| **Sensibilidad** | De los morosos que había, cuántos detecté | Cuando dejar pasar un caso malo es lo caro |
| **AUC** | Qué tan bien ordena los casos de peor a mejor, en todos los umbrales | Para comparar modelos sin fijar un umbral |

Precisión y sensibilidad se mueven en direcciones opuestas: si el modelo marca más casos como morosos,
detecta más (sube la sensibilidad) y se equivoca más (baja la precisión). El detalle se ve en un curso de
aprendizaje automático.